# Day 9 — Linear Algebra Intuition
**Objective:** Understand what vectors and matrices *do* geometrically, implement core operations from scratch, and verify against NumPy.

---


## 0. Imports

In [1]:
import numpy as np
import math


---
## Part 1 — Exercises

Work through each operation **by hand first** (in the markdown cells), then verify with NumPy.


### Exercise 1 — Dot Product

**By hand:**

Given `a = [2, 3, -1]` and `b = [4, -1, 2]`:

```
a · b = (2×4) + (3×-1) + (-1×2)
      =   8   +   -3   +   -2
      =   3
```

**Formula:** `a · b = Σ aᵢ × bᵢ`


In [ ]:
a = np.array([2, 3, -1])
b = np.array([4, -1, 2])

# Three equivalent ways
result_dot  = np.dot(a, b)
result_at   = a @ b
result_sum  = sum(ai * bi for ai, bi in zip(a, b))   # manual as sanity check

print(f"np.dot(a, b)  = {result_dot}")
print(f"a @ b         = {result_at}")
print(f"manual sum    = {result_sum}")
assert result_dot == result_at == result_sum, "Mismatch!"
print("All three methods agree")


np.dot(a, b)  = 3
a @ b         = 3
manual sum    = 3
✓ All three methods agree


### Exercise 2 — Matrix Multiplication

**By hand:**

```
A = [[1, 2],    B = [[5, 6],
     [3, 4]]         [7, 8]]

C[0][0] = row0(A) · col0(B) = 1×5 + 2×7 = 19
C[0][1] = row0(A) · col1(B) = 1×6 + 2×8 = 22
C[1][0] = row1(A) · col0(B) = 3×5 + 4×7 = 43
C[1][1] = row1(A) · col1(B) = 3×6 + 4×8 = 50

C = [[19, 22],
     [43, 50]]
```


In [ ]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

result_at     = A @ B
result_matmul = np.matmul(A, B)

print("A @ B =")
print(result_at)
print()
print("np.matmul(A, B) =")
print(result_matmul)
assert np.array_equal(result_at, result_matmul), "Mismatch!"
print("Both methods agree")


A @ B =
[[19 22]
 [43 50]]

np.matmul(A, B) =
[[19 22]
 [43 50]]
✓ Both methods agree


### Exercise 3 — Identity, Transpose, Inverse

- **Identity matrix:** Multiplying any matrix by I leaves it unchanged (like multiplying a number by 1).
- **Transpose:** Flip rows and columns — `A.T[i][j] = A[j][i]`.
- **Inverse:** `A⁻¹` such that `A @ A⁻¹ = I`. Only exists for square, non-singular matrices.


In [ ]:
A = np.array([[1, 2],
              [3, 4]])

# Identity
I = np.eye(2)
print("Identity matrix (2×2):")
print(I)
print()

# Transpose
print("A =");          print(A)
print("A.T =");        print(A.T)
print()

# Inverse
A_inv = np.linalg.inv(A)
print("A⁻¹ =");        print(A_inv)
print()

# Verify: A @ A⁻¹ ≈ I  (small float noise expected)
product = A @ A_inv
print("A @ A⁻¹ (should be ≈ identity):")
print(np.round(product, 10))
assert np.allclose(product, np.eye(2)), "Inverse check failed!"
print("A @ A⁻¹ = I confirmed")

Identity matrix (2×2):
[[1. 0.]
 [0. 1.]]

A =
[[1 2]
 [3 4]]
A.T =
[[1 3]
 [2 4]]

A⁻¹ =
[[-2.   1. ]
 [ 1.5 -0.5]]

A @ A⁻¹ (should be ≈ identity):
[[1. 0.]
 [0. 1.]]
✓ A @ A⁻¹ = I confirmed


---
## Part 2 — Implement From Scratch

No NumPy inside the implementations — pure Python loops only.


### 2a — `dot_product()` from scratch

**Algorithm:** multiply element-wise, sum the results.  
Raises `ValueError` for mismatched lengths.


In [5]:
def dot_product(a, b):
    """
    Compute the dot product of two 1-D vectors.
    
    Parameters
    ----------
    a, b : list or tuple of numbers, same length
    
    Returns
    -------
    float : scalar dot product
    
    Raises
    ------
    ValueError : if lengths differ
    TypeError  : if inputs are not sequences
    """
    if len(a) != len(b):
        raise ValueError(
            f"Vectors must have the same length, got {len(a)} and {len(b)}"
        )
    
    total = 0.0
    for ai, bi in zip(a, b):
        total += ai * bi
    return total


# ── Quick smoke test ──────────────────────────────────────────────────────────
a = [2, 3, -1]
b = [4, -1, 2]
print(f"dot_product({a}, {b}) = {dot_product(a, b)}")   # expected 3.0


dot_product([2, 3, -1], [4, -1, 2]) = 3.0


### 2b — `matmul()` from scratch

**Algorithm:** for each output cell `C[i][j]`, compute the dot product of  
row `i` from A and column `j` from B.  
Raises `ValueError` for incompatible shapes.


In [6]:
def matmul(A, B):
    """
    Multiply two 2-D matrices A (m×n) and B (n×p).
    
    Parameters
    ----------
    A : list of lists, shape (m, n)
    B : list of lists, shape (n, p)
    
    Returns
    -------
    C : list of lists, shape (m, p)
    
    Raises
    ------
    ValueError : if inner dimensions don't match
    """
    m  = len(A)
    n  = len(A[0])
    n2 = len(B)
    p  = len(B[0])
    
    if n != n2:
        raise ValueError(
            f"Incompatible shapes: ({m}×{n}) cannot multiply ({n2}×{p}). "
            f"Inner dimensions must match."
        )
    
    # Initialise output matrix with zeros
    C = [[0.0] * p for _ in range(m)]
    
    for i in range(m):
        for j in range(p):
            for k in range(n):
                C[i][j] += A[i][k] * B[k][j]
    return C


# ── Quick smoke test ──────────────────────────────────────────────────────────
A = [[1, 2], [3, 4]]
B = [[5, 6], [7, 8]]
C = matmul(A, B)
print("matmul result:")
for row in C: print(" ", row)   # expected [[19,22],[43,50]]


matmul result:
  [19.0, 22.0]
  [43.0, 50.0]


---
## Part 3 — Verification vs NumPy

Compare scratch implementations against NumPy on multiple cases.


### 3a — Dot product verification

In [7]:
def verify_dot(a, b, label=""):
    scratch = dot_product(a, b)
    numpy_  = float(np.dot(a, b))
    match   = math.isclose(scratch, numpy_, rel_tol=1e-9)
    status  = "✓" if match else "✗"
    print(f"{status} [{label}]  scratch={scratch}  numpy={numpy_}  match={match}")
    assert match, f"Mismatch on case: {label}"

# Normal cases
verify_dot([2, 3, -1], [4, -1, 2],    label="basic 3-element")
verify_dot([1, 0, 0],  [0, 1, 0],     label="orthogonal unit vectors → 0")
verify_dot([3, 4],     [3, 4],         label="vector with itself → magnitude²")
verify_dot([1.5, 2.5], [-1.0, 4.0],   label="floats")

# Edge cases
verify_dot([0, 0, 0],  [5, -3, 2],    label="zero vector → 0")
verify_dot([1000000],  [1000000],      label="large values")
verify_dot([-3, -2],   [-1, -4],       label="all negatives")

print()
print("All dot product verification tests passed ✓")


✓ [basic 3-element]  scratch=3.0  numpy=3.0  match=True
✓ [orthogonal unit vectors → 0]  scratch=0.0  numpy=0.0  match=True
✓ [vector with itself → magnitude²]  scratch=25.0  numpy=25.0  match=True
✓ [floats]  scratch=8.5  numpy=8.5  match=True
✓ [zero vector → 0]  scratch=0.0  numpy=0.0  match=True
✓ [large values]  scratch=1000000000000.0  numpy=1000000000000.0  match=True
✓ [all negatives]  scratch=11.0  numpy=11.0  match=True

All dot product verification tests passed ✓


### 3b — Matmul verification

In [8]:
def verify_matmul(A, B, label=""):
    C_scratch = matmul(A, B)
    C_numpy   = np.matmul(A, B).tolist()
    # Compare element-wise with tolerance
    flat_s = [x for row in C_scratch for x in row]
    flat_n = [x for row in C_numpy   for x in row]
    match  = all(math.isclose(s, n, rel_tol=1e-9) for s, n in zip(flat_s, flat_n))
    status = "✓" if match else "✗"
    print(f"{status} [{label}]  match={match}")
    assert match, f"Mismatch on case: {label}"

# Normal cases
verify_matmul([[1,2],[3,4]], [[5,6],[7,8]],         label="2×2 square")
verify_matmul([[1,2,3]],     [[4],[5],[6]],          label="1×3 @ 3×1 → 1×1 (dot product)")
verify_matmul([[1,2],[3,4]], [[1,0],[0,1]],          label="multiply by identity")
verify_matmul([[1,0],[0,1]], [[7,-3],[2,5]],         label="identity @ A = A")

# Non-square
verify_matmul([[1,2,3],[4,5,6]], [[7,8],[9,10],[11,12]], label="2×3 @ 3×2 → 2×2")

# Edge cases
verify_matmul([[0,0],[0,0]], [[5,6],[7,8]],          label="zero matrix")
verify_matmul([[2,0],[0,3]], [[1,0],[0,1]],          label="diagonal scaling matrix")

print()
print("All matmul verification tests passed ✓")


✓ [2×2 square]  match=True
✓ [1×3 @ 3×1 → 1×1 (dot product)]  match=True
✓ [multiply by identity]  match=True
✓ [identity @ A = A]  match=True
✓ [2×3 @ 3×2 → 2×2]  match=True
✓ [zero matrix]  match=True
✓ [diagonal scaling matrix]  match=True

All matmul verification tests passed ✓


### 3c — Error handling (edge cases)

In [9]:
# Test that errors are raised correctly

# dot_product: mismatched lengths
try:
    dot_product([1, 2, 3], [4, 5])
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ dot_product length mismatch caught: {e}")

# matmul: incompatible inner dimensions
try:
    matmul([[1, 2], [3, 4]], [[1, 2], [3, 4], [5, 6]])
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ matmul shape mismatch caught: {e}")

# dot_product: single-element vectors
result = dot_product([7], [6])
print(f"✓ Single-element dot product: 7·6 = {result}")

# matmul: 1×1 matrices
result = matmul([[5]], [[4]])
print(f"✓ 1×1 matmul: [[5]] @ [[4]] = {result}")


✓ dot_product length mismatch caught: Vectors must have the same length, got 3 and 2
✓ matmul shape mismatch caught: Incompatible shapes: (2×2) cannot multiply (3×2). Inner dimensions must match.
✓ Single-element dot product: 7·6 = 42.0
✓ 1×1 matmul: [[5]] @ [[4]] = [[20.0]]


---
## Part 4 — Geometric Meaning

### What does the dot product *mean* geometrically?

The dot product `a · b = |a| × |b| × cos(θ)` where `θ` is the angle between the vectors.

This means:
- **a · b > 0** → vectors point in roughly the same direction (angle < 90°)
- **a · b = 0** → vectors are *perpendicular* (orthogonal) — they share no direction
- **a · b < 0** → vectors point in opposite directions (angle > 90°)
- **a · b = |a|²** when `b = a` — the dot product of a vector with itself is its squared length

In machine learning: the dot product measures *similarity*. It's used in cosine similarity, attention mechanisms (queries × keys in transformers), and every linear layer's forward pass.


In [10]:
# Demonstrate geometric meaning of dot product

def angle_between(a, b):
    """Return angle in degrees between vectors a and b."""
    cos_theta = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    cos_theta = np.clip(cos_theta, -1.0, 1.0)   # guard against float errors
    return math.degrees(math.acos(cos_theta))

cases = [
    ([1, 0],  [1, 0],  "same direction"),
    ([1, 0],  [0, 1],  "perpendicular"),
    ([1, 0],  [-1, 0], "opposite direction"),
    ([1, 1],  [1, 0],  "45° apart"),
]

print(f"{'Case':<22} {'dot':>6}  {'angle':>8}")
print("-" * 42)
for a, b, label in cases:
    d = np.dot(a, b)
    theta = angle_between(a, b)
    print(f"{label:<22} {d:>6.2f}  {theta:>7.1f}°")


Case                      dot     angle
------------------------------------------
same direction           1.00      0.0°
perpendicular            0.00     90.0°
opposite direction      -1.00    180.0°
45° apart                1.00     45.0°


### What does matrix multiplication *mean* geometrically?

A matrix is a **linear transformation** — it warps space while keeping:
1. The origin fixed
2. All lines straight (no curving)
3. Parallel lines still parallel

The **columns of a matrix** tell you exactly where the basis vectors land after transformation:
- Column 1 → where `î = [1, 0]` goes
- Column 2 → where `ĵ = [0, 1]` goes

To find where *any* vector goes: express it as a combination of basis vectors, apply the same scaling.

**Matrix multiplication `A @ B`** means: first apply transformation B, then apply transformation A. Order matters — `AB ≠ BA`.

In neural networks: each weight matrix is a linear transformation (rotation, scaling, projection). A deep network is just many of these chained together.


In [11]:
# Demonstrate matrix as geometric transformation

# Rotation by 90 degrees counter-clockwise
# i-hat [1,0] goes to [0,1]; j-hat [0,1] goes to [-1,0]
theta = math.radians(90)
R = np.array([[math.cos(theta), -math.sin(theta)],
              [math.sin(theta),  math.cos(theta)]])

v = np.array([1, 0])
v_rotated = R @ v

print("Rotation matrix (90 deg CCW):")
print(np.round(R, 4))
print("v =", v, "  (pointing right)")
print("R @ v =", np.round(v_rotated, 4), "  (now pointing up)")
print()

# Scaling: stretch x by 2, y by 3
S = np.array([[2, 0],
              [0, 3]])
u = np.array([1, 1])
print("Scaling matrix:")
print(S)
print("u =", u)
print("S @ u =", S @ u, "  (x doubled, y tripled)")
print()

# Composition: first rotate, then scale
composed = S @ R
print("Composed (scale after rotate) = S @ R:")
print(np.round(composed, 4))
print("Composed @ v =", np.round(composed @ v, 4))
print("(First rotated to [0,1], then y was tripled to [0,3])")


Rotation matrix (90 deg CCW):
[[ 0. -1.]
 [ 1.  0.]]
v = [1 0]   (pointing right)
R @ v = [0. 1.]   (now pointing up)

Scaling matrix:
[[2 0]
 [0 3]]
u = [1 1]
S @ u = [2 3]   (x doubled, y tripled)

Composed (scale after rotate) = S @ R:
[[ 0. -2.]
 [ 3.  0.]]
Composed @ v = [0. 3.]
(First rotated to [0,1], then y was tripled to [0,3])


---
## Part 5 — Summary Table

| Operation | Formula | NumPy | Geometric meaning |
|-----------|---------|-------|-------------------|
| Dot product | `Σ aᵢbᵢ` | `np.dot(a,b)` or `a @ b` | Similarity / projection |
| Matrix × vector | `Σ Aᵢₖ vₖ` | `A @ v` | Transform a vector |
| Matrix × matrix | `Σ Aᵢₖ Bₖⱼ` | `A @ B` | Compose two transformations |
| Transpose | `Aᵀ[i][j] = A[j][i]` | `A.T` | Flip rows ↔ columns |
| Inverse | `A @ A⁻¹ = I` | `np.linalg.inv(A)` | Undo a transformation |
| Identity | Diagonal of 1s | `np.eye(n)` | Do nothing (like × 1) |


---
## Self-Review & Reflection

**What I learned:**
- Vectors are arrows in space, not just lists — this reframe makes matrix ops intuitive
- The dot product measures directional similarity: zero means perpendicular
- Matrix columns encode where basis vectors land — this is the key to understanding any transform
- Matrix multiplication is function composition, applied right-to-left
- Implementing from scratch exposes the triple nested loop in matmul — O(n³) complexity

**What was difficult:**
- Keeping track of row-vs-column indexing in the scratch matmul
- Floating point noise in inverse verification (solved with `np.allclose`)

**What I'd explore next:**
- Determinant: what does it mean geometrically? (Area scaling factor)
- Eigenvalues/eigenvectors: special vectors that only get scaled, not rotated
- How these operations map directly to a neural network's forward pass
